# Setup Layer (UC schemas + volumes)

**Run this notebook once per workspace** before the data notebooks.

It provisions the Unity Catalog schemas and volumes used by the rest of the pipeline:

| Schema | Volume | Purpose |
|---|---|---|
| `bronze` | `landing` | Raw CSVs uploaded by hand (or CLI) before `01_ingestion_layer` runs |
| `bronze` | `_meta` | Auto Loader checkpoints and inferred-schema logs |
| `silver` | `outputs` | Parquet files written by cleaning + feature-engineering layers |
| `gold` | `outputs` | Final/curated datasets for analytics |

All DDL is **idempotent** (`IF NOT EXISTS`) — safe to re-run.

**Prerequisite:** the catalog (`CATALOG` below) must already exist. Creating catalogs typically requires workspace-admin permission, so change the constant if your workspace uses a different catalog name.

## 1. Config

Change `CATALOG` if your workspace uses a different name (e.g. `main`, `students`, `sandbox`). Every other notebook in this pipeline has the same constant — keep them in sync.

In [0]:
CATALOG = "crime_data"

print(f"Using catalog: {CATALOG}")

## 2. Create schemas and volumes

Backticks around `` `_meta` `` are required because UC identifiers starting with `_` must be quoted.

In [0]:
# Schemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

# Bronze volumes — landing CSVs + Auto Loader state
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.landing")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.`_meta`")

# Silver volume — parquet outputs from cleaning + feature engineering
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.silver.outputs")

# Gold volume — final curated datasets
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.gold.outputs")

print("Schemas and volumes provisioned.")

## 3. Verify

In [0]:
for schema in ["bronze", "silver", "gold"]:
    print(f"\n{CATALOG}.{schema} — volumes:")
    spark.sql(f"SHOW VOLUMES IN {CATALOG}.{schema}").show(truncate=False)

## Next steps

1. Upload the bronze CSVs to `/Volumes/<catalog>/bronze/landing/` (see `documentation/ingestion_layer.md` for the expected folder layout).
2. Run `01_ingestion_layer.ipynb` → `02_cleaning_and_validation_layer.ipynb` → `03_feature_engineering_and_transformation.ipynb` in order.

**Permission errors?** If any `CREATE SCHEMA` / `CREATE VOLUME` failed, you likely lack `CREATE SCHEMA` or `CREATE VOLUME` privileges on the catalog. Ask a workspace admin to run this notebook once on your behalf, or to grant you those privileges.